# Optimisation complète par ZONE — analyse consciente de l'architecture

Pipeline final : pour chaque modèle, on optimise **uniquement la zone statique**
(backbone + FPN) et on laisse le décodage/NMS en eager. C'est ce qui débloque le
vrai gain (le NMS à shapes dynamiques tuait TRT/compile/cudagraphs sur le modèle
complet). Pour chaque variante : **benchmark + MAP@640 + profiling** (avant/après),
et **tout est sauvegardé** sous un préfixe (Google Drive sur Colab).

**Modèles** : RetinaNet R50, FCOS R50, EfficientDet D4/D5/D6  *(R101 abandonné)*

**Variantes** (zone statique optimisée, NMS eager)

| Variante | Zone optimisée comment | MAP ? | Profil ? |
|---|---|:---:|:---:|
| `baseline` | — (modèle complet FP32) | ✓ | ✓ |
| `fp16` | autocast **modèle complet** (Tensor Cores) | ✓ | ✓ |
| `zone_torchscript` | backbone → TorchScript (fusion Conv-BN) | ✗ | ✓ |
| `zone_compile` | backbone → torch.compile (inductor/cudagraphs) | ✗ | ✓ |
| `zone_cudagraphs` | backbone → CUDA graphs | ✗ | ✓ |
| `zone_trt_fp16` | backbone → TensorRT FP16 | ✓ | ✓ |
| `zone_trt_int8` | backbone → TensorRT INT8 + calibration | ✓ | ✓ |

> Pourquoi FP16 est « modèle complet » et pas « zone » : l'autocast doit envelopper
> tout le forward, sinon les features FP16 fuient vers la tête FP32 (mismatch de dtype).
>
> Pourquoi MAP seulement sur certaines : seules les variantes qui **changent la
> précision** (fp16, trt) peuvent bouger la MAP ; les autres sont FP32-équivalentes.

## 0. Installation (Colab)
```bash
!pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet
```
TensorRT/CUDA sont pré-installés dans le runtime Colab GPU.

In [1]:
# Colab — décommenter à la première exécution
# !pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet

## 1. Préfixe de sortie (Google Drive)
Tous les artefacts (résultats, logs, profils, métriques) seront écrits sous ce préfixe.
En local il est vide ; sur Colab il pointe vers `MyDrive/ProjectDSAI2026_2` (créé au besoin).

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # racine du projet

from optimizations.paths import set_prefix, describe

# Monter le Drive sur Colab et fixer le préfixe
try:
    from google.colab import drive
    drive.mount("/content/drive")
    set_prefix("/content/drive/MyDrive/ProjectDSAI2026_2")
except Exception:
    set_prefix("./out")

print(describe())

Préfixe de sortie : ./out


## 2. Imports, capacités & configuration

In [3]:
from datetime import datetime
import torch
import pandas as pd

import optimizations as opt
from optimizations import OptimizationRunner, RunConfig, ModelSpec, DEFAULT_VARIANTS

from utils.data_loader import load_profiling_data, load_eval_data

CAPS = opt.print_report()

# ── Paramètres expérience ─────────────────────────────────────────────────────
N_WARMUP   = 50
N_MEASURE  = 1000
N_PROFILE  = 150       # itérations actives du profiler (pas de trace → léger)
N_PROFILE_DATA = 2000  # images chargées pour benchmark + profiling
N_EVAL     = 2000      # images pour la MAP@640
DO_PROFILE = True
DO_INT8    = False     # True → active TensorRT INT8 (calibration ~5 min/modèle)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

IMG_DIR  = "datasets/coco/val2017"
ANN_FILE = "datasets/coco/annotations/instances_val2017.json"

if DEVICE == "cpu":
    print("\n⚠ Aucun GPU CUDA — le benchmark nécessite CUDA (build torch +cuXXX).")

Plateforme   : Windows (local)
PyTorch      : 2.8.0+cu129
CUDA         : ✓ NVIDIA GeForce RTX 5060 Laptop GPU
Backend compile recommandé : cudagraphs

Techniques d'optimisation disponibles :
  FP16 autocast               ✓ disponible
  TorchScript                 ✓ disponible
  torch.compile (cudagraphs)  ✓ disponible
    └ inductor (Triton)       ✗ indisponible
  ONNX export                 ✓ disponible
  ONNX Runtime                ✓ disponible
  TensorRT FP16/INT8          ✗ indisponible


## 3. Données

In [4]:
from pycocotools.coco import COCO

profile_data = load_profiling_data(IMG_DIR, ANN_FILE, n=max(N_PROFILE_DATA, N_WARMUP+N_MEASURE))
eval_data, _ = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL)
coco_gt      = COCO(ANN_FILE)
print(f"Profilage/bench : {len(profile_data)} images")
print(f"Eval MAP        : {len(eval_data)} images")

loading annotations into memory...


JSONDecodeError: Extra data: line 1 column 19987841 (char 19987840)

## 4. Modèles et orchestrateur

R101 est abandonné. Les EfficientDet (`effdet`) sautent `zone_compile` par défaut
(inductor sur BiFPN peut être lent) — mettre `EFFDET_WITH_COMPILE=True` pour l'activer.

In [ ]:
import models.retinanet_r50  as r50
import models.fcos_r50        as fcos
import models.efficientdet_d4 as d4
import models.efficientdet_d5 as d5
import models.efficientdet_d6 as d6

SPECS = {
    "retinanet_r50":   ModelSpec("retinanet_r50",   r50,  "torchvision", has_map=True),
    "fcos_r50":        ModelSpec("fcos_r50",        fcos, "torchvision", has_map=True),
    "efficientdet_d4": ModelSpec("efficientdet_d4", d4,   "effdet",      has_map=True),
    "efficientdet_d5": ModelSpec("efficientdet_d5", d5,   "effdet",      has_map=True),
    "efficientdet_d6": ModelSpec("efficientdet_d6", d6,   "effdet",      has_map=True),
}

# ── Sélection des variantes — priorité au SPEEDUP qui marche, puis TRT (prof) ──
def pick(names):
    return [v for v in DEFAULT_VARIANTS if v.name in names]

# torchvision : fp16 + compile_fp16 (les gagnants T4 : ×1.24, ×2.35) + TRT backbone
VARIANTS_TV = pick([
    "baseline", "fp16", "compile_fp16",
    "zone_trt_fp16",                          # TensorRT (exigence de l'encadrant)
    "zone_cudagraphs", "zone_torchscript",   # comparses (gain modeste sur T4)
])

# effdet : pas de compile_fp16 (inductor lent sur BiFPN) ; TRT folded à la place
VARIANTS_EFFDET = pick([
    "baseline", "fp16",
    "zone_trt_fp16", "zone_trt_folded",      # TensorRT (folded = solution BiFPN)
    "zone_cudagraphs", "zone_torchscript",
])

print("torchvision :", [v.name for v in VARIANTS_TV])
print("effdet      :", [v.name for v in VARIANTS_EFFDET])

In [ ]:
RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUBDIR = f"results/optimization/{RUN_ID}"

config = RunConfig(
    n_warmup=N_WARMUP, n_measure=N_MEASURE, n_profile=N_PROFILE,
    do_profile=DO_PROFILE, device=DEVICE,
    compile_backend=CAPS.compile_backend,
    trt_available=CAPS.flags["tensorrt"], do_int8=DO_INT8,
)
runner = OptimizationRunner(profile_data, eval_data, coco_gt, config, RUN_SUBDIR)
print("Sorties →", runner.run_dir)

## 5. Exécution — un modèle par cellule

Chaque cellule sauvegarde au fur et à mesure (et fige les résultats à la fin du
modèle). Si une cellule plante/déconnecte, les résultats déjà écrits sur le Drive
sont conservés. Relancer une cellule réexécute juste ce modèle.

In [ ]:
runner.run_model(SPECS["retinanet_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["fcos_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["efficientdet_d4"], VARIANTS_EFFDET)

In [ ]:
runner.run_model(SPECS["efficientdet_d5"], VARIANTS_EFFDET)

In [ ]:
runner.run_model(SPECS["efficientdet_d6"], VARIANTS_EFFDET)

## 6. Résultats agrégés

In [ ]:
df = runner.to_dataframe()
print("Statuts :", dict(df["status"].value_counts()))
df

In [ ]:
spd = runner.speedup_table()
spd.to_csv(runner.run_dir / "speedup.csv", index=False)
spd

In [ ]:
fails = df[df["status"]=="FAILED"][["model","variant","error"]]
print("Tracebacks complets dans", runner.run_dir/"errors")
print("Aucun échec.") if len(fails)==0 else display(fails)


## 7. Profiling avant/après — comment les opérations ont changé

Compare la table d'opérations du `baseline` et d'une variante optimisée : on voit
la **fusion** (moins d'opérations distinctes, temps GPU concentré) après TRT/cudagraphs.

In [ ]:
# Synthèse profiling avant/après — SAUVEGARDÉE (le display est secondaire).
# Mesure clé : le nombre d'opérations distinctes chute après fusion (TRT/cudagraphs),
# et le temps GPU se concentre. On agrège (modèle × variante) et on sauvegarde.
def load_prof(model, variant):
    p = runner.run_dir / "profiles" / f"{model}_{variant}.csv"
    return pd.read_csv(p) if p.exists() else None

rows = []
for model in SPECS:
    for v in [s.name for s in DEFAULT_VARIANTS]:
        d = load_prof(model, v)
        if d is None:
            continue
        rows.append({
            "model":         model,
            "variant":       v,
            "n_ops":         len(d),                              # chute = fusion
            "cuda_us_total": round(float(d["cuda_us_total"].sum()), 1),
            "top_op":        str(d.iloc[0]["op"]) if len(d) else "",
        })
prof_summary = pd.DataFrame(rows)
prof_summary.to_csv(runner.run_dir / "profiling_summary.csv", index=False)
print("Sauvegardé →", runner.run_dir / "profiling_summary.csv")
prof_summary

## 8. Décomposition par module (baseline)

In [ ]:
def load_mod(model, variant="baseline"):
    p = runner.run_dir / "modules" / f"{model}_{variant}.csv"
    return pd.read_csv(p) if p.exists() else None

rows = []
for model in SPECS:
    dfm = load_mod(model)
    if dfm is None: continue
    total = dfm["mean_ms"].sum()
    def part(pat): return dfm[dfm["type"].str.contains(pat, na=False, regex=True)]["mean_ms"].sum()
    conv, bn, act = part("Conv"), part("BatchNorm|FrozenBatch"), part("ReLU|SiLU|Swish")
    rows.append({"model":model,"total_ms":round(total,2),
                 "Conv_%":round(100*conv/total,1) if total else 0,
                 "BN_%":round(100*bn/total,1) if total else 0,
                 "Act_%":round(100*act/total,1) if total else 0,
                 "CBR_fusible_%":round(100*(conv+bn+act)/total,1) if total else 0})
df_break = pd.DataFrame(rows)
df_break.to_csv(runner.run_dir/"module_breakdown.csv", index=False)
df_break

## 9. Vue transversale — opérations les plus accélérables

(Point 2 du cahier des charges) Agrège les profils baseline de tous les modèles
par opération : quelles briques dominent le temps GPU et la mémoire, tous modèles confondus.

In [ ]:
frames = []
for model in SPECS:
    d = load_prof(model, "baseline")
    if d is not None:
        d = d.copy(); d["model"] = model
        frames.append(d)

if frames:
    allops = pd.concat(frames, ignore_index=True)
    by_op = (allops.groupby("op")
             .agg(cuda_us=("cuda_us_total","sum"),
                  self_cuda_mem=("self_cuda_mem","sum"),
                  count=("count","sum"))
             .sort_values("cuda_us", ascending=False).round(1))
    by_op.to_csv(runner.run_dir/"ops_transversal.csv")
    print("Opérations par temps GPU cumulé (tous modèles) :")
    display(by_op.head(20))
else:
    print("Pas de profils — exécuter les modèles d'abord.")

## 10. Tout est sauvegardé

```
<préfixe>/results/optimization/<RUN_ID>/
  run.log                       journal complet
  results.csv / results_final.csv
  speedup.csv
  module_breakdown.csv  ops_transversal.csv
  bench/<model>_<variant>.json      vitesse brute
  eval/<model>_<variant>.json       MAP/AR COCO complètes
  modules/<model>_<variant>.csv     timing par module (baseline/fp16)
  profiles/<model>_<variant>.csv    kernels/mémoire/opérations (avant/après)
  errors/<model>_<variant>.txt
```
Sur Colab, ce dossier est directement sur ton Drive → rien à recopier.

In [ ]:
print("Tous les artefacts sont dans :", runner.run_dir)
for sub in ["bench","eval","modules","profiles","errors"]:
    d = runner.run_dir / sub
    n = len(list(d.glob('*'))) if d.exists() else 0
    print(f"  {sub:10s}: {n} fichiers")